# Topic 3: Matrix Multiplication (The Crossover Point)

Welcome to the final topic of this section! We have established that GPUs excel at massive parallel throughput (Topic 1) and that moving data over the PCIe bus is slow (Topic 2).

Now let's look at **Matrix Multiplication** ($C = A \times B$). Why is this operation the absolute sweetheart of GPU acceleration? And at what size does the GPU actually begin to beat the CPU?

### What We Will Learn
*   **MatMul Arithmetic Intensity:** Why matrix multiplication has high FLOPs-to-byte ratio.
*   **The Crossover Point:** Discover the matrix size threshold where GPU wins.
*   **Benchmark sweep:** Compare CPU vs GPU timings across varying dimensions.

### The Backdrop
For two $N \times N$ matrices, the memory footprint (bytes to transfer) grows quadratically ($O(N^2)$), but the arithmetic work (total floating-point operations) grows cubically ($O(N^3)$).

This means as $N$ grows, the arithmetic intensity increases: we perform more math per byte of data we fetch. This cubic growth is what makes large matrix multiplications perfect candidates for GPUs.

### Visualizing the Crossover Point

Below is a whiteboard sketch illustrating the execution times of CPU vs. GPU. Notice that for small matrices, the CPU is faster because it has zero launch latency, but as the matrix size grows, the CPU time explodes while the GPU handles the parallel throughput effortlessly:

![MatMul Crossover Point](images/matmul-crossover.svg)

### Step 1: Initialize Libraries

We'll import PyTorch and time to run our benchmark sweep.

In [ ]:
import time  # For wall-clock timing
import torch  # For tensor operations
device = "cuda" if torch.cuda.is_available() else "cpu"

Now, let's write a benchmarking function that runs matrix multiplications of size $N \times N$ on both CPU and GPU.

### Step 2: Benchmarking Small Matrices ($32 \times 32$)

We will benchmark a small matrix multiplication. Let's time it on both CPU and GPU.

In [ ]:
# Benchmark 32x32 matrix multiplication on CPU vs GPU
x_cpu, y_cpu = torch.randn(32, 32), torch.randn(32, 32)
t0 = time.perf_counter()  # Start CPU timer
_ = torch.matmul(x_cpu, y_cpu)  # Run on CPU
cpu_time = time.perf_counter() - t0
print(f"CPU time (32x32): {cpu_time:.6f}s")

In [ ]:
# GPU benchmark for 32x32 with synchronization
x_gpu, y_gpu = x_cpu.to(device), y_cpu.to(device)
if device == "cuda":
    torch.cuda.synchronize()  # Clear GPU stream
t0 = time.perf_counter()  # Start CPU timer
_ = torch.matmul(x_gpu, y_gpu)  # Run on GPU
if device == "cuda":
    torch.cuda.synchronize()  # Wait for completion
print(f"GPU time (32x32): {time.perf_counter() - t0:.6f}s")

Compare the two numbers. For a tiny matrix size like $32 \times 32$, the CPU is often **faster** than the GPU. Why? Because the time spent setting up and launching the GPU kernel is larger than the actual math work. The overhead dominates.

### Step 3: Benchmarking Large Matrices ($2048 \times 2048$)

Let's sweep to the opposite end of the spectrum: a large $2048 \times 2048$ matrix multiply.

In [ ]:
# Benchmark 2048x2048 matrix multiplication on CPU vs GPU
x_cpu, y_cpu = torch.randn(2048, 2048), torch.randn(2048, 2048)
t0 = time.perf_counter()  # Start CPU timer
_ = torch.matmul(x_cpu, y_cpu)  # Run on CPU
cpu_time = time.perf_counter() - t0
print(f"CPU time (2048x2048): {cpu_time:.4f}s")

In [ ]:
# GPU benchmark for 2048x2048 with synchronization
x_gpu, y_gpu = x_cpu.to(device), y_cpu.to(device)
if device == "cuda":
    torch.cuda.synchronize()  # Clear GPU stream
t0 = time.perf_counter()  # Start CPU timer
_ = torch.matmul(x_gpu, y_gpu)  # Run on GPU
if device == "cuda":
    torch.cuda.synchronize()  # Wait for completion
print(f"GPU time (2048x2048): {time.perf_counter() - t0:.4f}s")

Now look at the numbers. The CPU time skyrocketed because the cubic $O(N^3)$ complexity hit it hard. But the GPU calculated it in a tiny fraction of the time. The thousands of parallel GPU ALUs masked the launch latency and completely ran away with the victory.

## First-Principles Checkpoint: Crossover Point

This experiment shows that the statement "GPUs are faster than CPUs" is false without context. 

*   For **small matrices ($N < 128$)**, the CPU wins because it avoids GPU kernel launch latency and PCIe transfer overhead.
*   For **large matrices ($N \\ge 256$)**, the arithmetic complexity ($O(N^3)$) allows the GPU to utilize its high parallel throughput, making it the clear winner.\n
The threshold size where the GPU becomes faster is called the **crossover point**.

### Role Lens: Why it matters in practice
*   **DevOps / MLOps:** Batch sizing is everything. Running small batches (e.g., batch size 1) for real-time inference on a GPU is highly inefficient because it keeps the GPU in the low-crossover regime. Grouping requests into larger batches pushes the workload into the GPU-optimal throughput zone.
*   **Data Science:** When designing custom architectures (like light-weight MLP layers), keep matrix dimensions in multiples of 8 or 16 (preferably 64/128) to hit optimal alignments on GPU cores and exceed the crossover threshold.
*   **Data Engineering:** If you are processing tabular datasets with tiny rows, running them on GPU accelerators is a waste. Process them on CPU unless you can bundle millions of rows together to make the GPU startup cost worthwhile.

Congratulations! You have completed Section 01: GPU Fundamentals. You now understand the physics of CPU vs GPU architecture, PCIe bottlenecks, and the crossover point. In the next section, we will start writing custom code directly to the hardware using CUDA without fear!